In [0]:
%run ./config

In [0]:
%run ./silver_utils

In [0]:
%run ../connection

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import ( DoubleType, IntegerType, StringType, TimestampType, DateType )
 
 adls_name = "adlsnewhp1"
 init_silver_config(adls_name)
 
 logger = get_logger("silver_global_stats")
 
 logger.info("=" * 70)
 logger.info("Silver: global_stats — START")
 logger.info(f"  Run ID    : {SilverConfig.RUN_ID}")
 logger.info(f"  Bronze in : {BronzePaths.GLOBAL}")
 logger.info(f"  Silver out: {SilverPaths.GLOBAL_STATS}")
 logger.info("=" * 70)

In [0]:


bronze_df, max_bronze_ts = read_bronze_incremental(
    spark, BronzePaths.GLOBAL , WatermarkPaths.GLOBAL,
    WatermarkPaths.WATERMARK_TABLE, logger
)

assert_required_columns(
    bronze_df,
    required_cols=["payload", "pipeline_run_id"],
    table_name="global_raw",
    logger=logger
)

logger.info(f"  Bronze schema: {bronze_df.schema.simpleString()}")

In [0]:


logger.info("CELL 3: Extracting global stats fields from payload.data")

typed_df = (
    bronze_df
    .select(
        
        F.from_unixtime(
            F.col("payload.data.updated_at").cast("long")
        ).cast(TimestampType()).alias("stats_timestamp"),

        F.to_date(
            F.from_unixtime(F.col("payload.data.updated_at").cast("long"))
        ).alias("stats_date"),                          

        
        F.col("payload.data.total_market_cap.usd")
         .cast(DoubleType())
         .alias("total_market_cap_usd"),

        
        F.col("payload.data.total_volume.usd")
         .cast(DoubleType())
         .alias("total_volume_24h_usd"),

        
        F.col("payload.data.market_cap_percentage.btc")
         .cast(DoubleType())
         .alias("btc_dominance_pct"),

        F.col("payload.data.market_cap_percentage.eth")
         .cast(DoubleType())
         .alias("eth_dominance_pct"),

     
        F.col("payload.data.active_cryptocurrencies")
         .cast(IntegerType())
         .alias("active_cryptos"),

        
        # Negative = market losing value (bearish), positive = gaining (bullish)
        F.col("payload.data.market_cap_change_percentage_24h_usd")
         .cast(DoubleType())
         .alias("market_cap_change_pct_24h"),

        
        F.col("pipeline_run_id").cast(StringType()),
    )
    .withColumn("silver_processed_timestamp", get_silver_timestamp(SilverConfig.RUN_TS))
)

raw_count = typed_df.count()
logger.info(f"  Rows after field extraction (expect 1 per run): {raw_count:,}")



In [0]:

# DATA QUALITY FILTERS
#
# GLOBAL-SPECIFIC RULES:
#   stats_date must not be null (no date = can't place this on timeline)
#   total_market_cap_usd must be positive (sanity: the global market has value)
#   btc_dominance_pct must be in [0, 100] (it's a percentage)
#   active_cryptos must be >= MIN_ACTIVE_CRYPTOS


logger.info("CELL 4: Applying data quality filters")

clean_df = (
    typed_df
    .filter(F.col("stats_date").isNotNull())
    .filter(F.col("stats_timestamp").isNotNull())
    .filter(
        F.col("total_market_cap_usd").isNotNull() &
        (F.col("total_market_cap_usd") > 0.0)
    )
    .filter(
        F.col("btc_dominance_pct").isNull() |
        (
            (F.col("btc_dominance_pct") >= DataQuality.MIN_BTC_DOMINANCE_PCT) &
            (F.col("btc_dominance_pct") <= DataQuality.MAX_BTC_DOMINANCE_PCT)
        )
    )
    .filter(
        F.col("active_cryptos").isNull() |
        (F.col("active_cryptos") >= DataQuality.MIN_ACTIVE_CRYPTOS)
    )
)

clean_count = clean_df.count()
logger.info(f"  Rows after quality filters: {clean_count:,}")

validate_drop_rate(
    rows_before  = raw_count,
    rows_after   = clean_count,
    max_fraction = DataQuality.MAX_DROP_FRACTION,
    table_name   = "global_stats",
    logger       = logger,
)

In [0]:


logger.info("CELL 5: Deduplicating on stats_date")

dedup_df    = clean_df.dropDuplicates(MergeKeys.GLOBAL_STATS)
dedup_count = dedup_df.count()
logger.info(f"  Rows after dedup: {dedup_count:,} (dropped {clean_count - dedup_count:,})")


In [0]:



final_df = dedup_df.select(*SilverColumns.GLOBAL_STATS)
log_schema(final_df, "global_stats_final", logger)

In [0]:


logger.info("CELL 7: MERGE into silver/global_stats")

merge_stats = delta_merge(
    spark      = spark,
    new_df     = final_df,
    table_path = SilverPaths.GLOBAL_STATS,
    merge_keys = MergeKeys.GLOBAL_STATS,
    logger     = logger,
)

update_watermark(
    spark          = spark,
    source_table   = WatermarkPaths.GLOBAL,
    new_ts         = max_bronze_ts,
    watermark_path = WatermarkPaths.WATERMARK_TABLE,
    run_ts         = SilverConfig.RUN_TS,
    logger         = logger,
)
 


In [0]:


logger.info("CELL 8: OPTIMIZE silver/global_stats")

spark.sql(f"OPTIMIZE delta.`{SilverPaths.GLOBAL_STATS}` ZORDER BY (stats_date)")
logger.info("  ✓ OPTIMIZE complete")


In [0]:






summary = {
    "notebook"                 : "02e_silver_global_stats",
    "pipeline_run_id"          : SilverConfig.RUN_ID,
    "run_timestamp_utc"        : SilverConfig.RUN_TS.isoformat(),
    "bronze_source"            : BronzePaths.GLOBAL,
    "silver_target"            : SilverPaths.GLOBAL_STATS,
    "rows_from_bronze"         : raw_count,
    "rows_after_quality_filter": clean_count,
    "rows_after_dedup"         : dedup_count,
    "merge_rows_before"        : merge_stats["rows_before"],
    "merge_rows_after"         : merge_stats["rows_after"],
    "merge_rows_inserted"      : merge_stats["rows_inserted"],
    "status"                   : "SUCCESS",
}

write_run_log(summary, LogPaths.GLOBAL_STATS, logger)

logger.info("=" * 70)
logger.info("Silver: global_stats — COMPLETE")
for k, v in summary.items():
    logger.info(f"  {k:<35}: {v}")
logger.info("=" * 70)

dbutils.notebook.exit(json.dumps(summary))
